# Unit 6 — Build a conditional VAE

A conditional variational autoencoder (CVAE) encodes a condition and target into a latent distribution. A decoder uses the condition and a latent sample to predict the target. For robot learning, the target is an action chunk. The condition can be robot state and camera features.

You will implement an encoder, the reparameterization operation, a conditional decoder, and the training and inference paths. Basic `nn.Linear` and activation layers are supplied by PyTorch. The exercises use small vectors first; the final opt-in run uses prepared ACT demonstration chunks. It does not download data.

The ELBO objective combines reconstruction error and KL divergence. Use the action mask from Unit 3 for padded chunks. Reading: [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), plus the [ACT paper](https://arxiv.org/abs/2304.13705) for its later Transformer posterior.

**PyTorch note:** `torch.randn_like(mu)` makes a random tensor with the same shape and device. Multiplication and addition keep gradients, so the encoder learns through the sample.

This notebook contains the full implementation.


In [ ]:
from pathlib import Path
import os, sys

def find_workspace():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "manifests" / "act_references.yaml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside this repository.")

WORKSPACE = find_workspace()
sys.path.insert(0, str(WORKSPACE / "src"))
os.environ.setdefault("HF_HUB_CACHE", str(WORKSPACE / ".local-data/hf/hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(WORKSPACE / ".local-data/hf/datasets"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".local-data/matplotlib"))
import torch
from torch import nn
import matplotlib.pyplot as plt
print("Workspace:", WORKSPACE)
print("PyTorch:", torch.__version__)


In [ ]:
torch.manual_seed(4)
CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN = 3, 2, 2, 8
condition = torch.randn(5, CONDITION_DIM)
target = torch.randn(5, TARGET_DIM)


## Exercises

Complete reference implementations.

## Exercise 1 — Conditional encoder (about 15 min)

Implement `ConditionalEncoder`. Join condition and target on the last axis. The network returns `mu` and `logvar`, each with shape `(..., latent_dim)`. The exercise fixture uses `(B,C)` and `(B,Y)` inputs.

**Method:** map the concatenated values through a small shared network, then use separate linear heads for mean and log variance. Clamp `logvar` to a stable range.

**Pointers:** `torch.cat(..., dim=-1)` joins features while preserving the batch axis.

<details><summary>More hints</summary>

Use two hidden linear layers with GELU. Set input width to `condition_dim + target_dim`; set both heads to `latent_dim`.

</details>

In [ ]:
class ConditionalEncoder(nn.Module):
    def __init__(self, condition_dim, target_dim, latent_dim, hidden_dim=128, mask_dim=0):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(condition_dim + target_dim + mask_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU())
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, condition, target, target_mask=None):
        if condition.shape[:-1] != target.shape[:-1]: raise ValueError("condition and target leading shapes must match")
        inputs = [condition, target]
        if target_mask is not None: inputs.append(target_mask.to(target.dtype))
        hidden = self.body(torch.cat(inputs, dim=-1))
        return self.mu(hidden), self.logvar(hidden).clamp(-20.0, 10.0)

In [ ]:
encoder = ConditionalEncoder(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
mu, logvar = encoder(condition, target)
assert mu.shape == logvar.shape == (5, LATENT_DIM), "Both Gaussian parameter outputs need shape (B, latent_dim)."
assert torch.isfinite(logvar).all(), "Log variance must be finite."
print("Encoder output shapes:", mu.shape, logvar.shape)

In [ ]:
print("Use: the encoder output feeds the reparameterization function in Exercise 2.")

## Exercise 2 — Reparameterization (about 15 min)

Implement `reparameterize(mu, logvar, noise=None)`. It returns a sample with shape `(B,L)`. When noise is supplied, use it instead of drawing random values. This makes the check reproducible.

Formula: `z = mu + exp(0.5 * logvar) * noise`.

**Pointers:** Keep `mu` and `logvar` in the calculation graph.

<details><summary>More hints</summary>

If `noise is None`, call `torch.randn_like(mu)`. Check the provided noise shape before using it.

</details>

In [ ]:
def reparameterize(mu, logvar, *, noise=None):
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar must have matching shapes")
    if noise is None:
        noise = torch.randn_like(mu)
    if noise.shape != mu.shape:
        raise ValueError("noise must have the same shape as mu")
    return mu + torch.exp(0.5 * logvar) * noise

In [ ]:
mu_check = torch.tensor([[1.0, -1.0]], requires_grad=True)
logvar_check = torch.zeros_like(mu_check, requires_grad=True)
z = reparameterize(mu_check, logvar_check, noise=torch.tensor([[2.0, 0.5]]))
torch.testing.assert_close(z, torch.tensor([[3.0, -0.5]]))
z.sum().backward()
assert torch.equal(mu_check.grad, torch.ones_like(mu_check)), "The sample must keep its gradient to mu."
assert torch.equal(logvar_check.grad, torch.tensor([[1.0, 0.25]])), "The sample must keep its gradient to logvar."
print("Sample:", z.detach())

In [ ]:
print("Use: the training path samples from the encoder; inference samples from a standard normal prior.")

## Exercise 3 — Conditional decoder (about 15 min)

Implement `ConditionalDecoder`. It takes condition `(..., C)` and latent `(..., L)` and returns a prediction `(..., Y)`. The output width is the target width.

**Pointers:** Join the condition and latent code along the final dimension. Keep all leading dimensions.

<details><summary>More hints</summary>

A two-layer MLP with GELU is enough for this exercise. The last layer width is `target_dim`.

</details>

In [ ]:
class ConditionalDecoder(nn.Module):
    def __init__(self, condition_dim, latent_dim, target_dim, hidden_dim=128):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(condition_dim + latent_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, target_dim))
    def forward(self, condition, latent):
        if condition.shape[:-1] != latent.shape[:-1]:
            raise ValueError("condition and latent leading shapes must match")
        return self.network(torch.cat((condition, latent), dim=-1))

In [ ]:
decoder = ConditionalDecoder(CONDITION_DIM, LATENT_DIM, TARGET_DIM, HIDDEN)
prediction = decoder(condition, torch.zeros(5, LATENT_DIM))
assert prediction.shape == target.shape, "Decoder output must match the target shape."
assert torch.isfinite(prediction).all(), "Decoder output must be finite."
print("Decoder output:", prediction.shape)

In [ ]:
print("Use: Exercise 4 combines encoder, sample, and decoder into a CVAE model.")

## Exercise 4 — Training and prior inference paths (about 15 min)

Implement `ConditionalVAE`. `forward(condition, target, noise=...)` uses the target to form a posterior sample and returns `(prediction, mu, logvar)`. `infer(condition, noise=...)` samples from the standard normal prior and decodes without a target. For action chunks, set `target_shape=(T,D)` and pass `is_pad` to `forward`; zero padded target values before encoding and provide the time mask to the encoder. The caller can supply noise for a repeatable check.

**Pointers:** Keep posterior training and prior inference as separate methods. Store `latent_dim`.

<details><summary>More hints</summary>

In `forward`, call your encoder, reparameterize, and decode. In `infer`, create or accept a latent sample of shape `(*condition.shape[:-1], latent_dim)`.

</details>

In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(self, condition_dim, target_dim, latent_dim, hidden_dim=128, *, target_shape=None):
        super().__init__()
        self.latent_dim, self.target_shape = latent_dim, target_shape
        target_width = target_dim
        mask_width = 0
        if target_shape is not None:
            target_width = 1
            for size in target_shape: target_width *= size
            if target_width != target_dim: raise ValueError("target_dim must equal product of target_shape")
            if len(target_shape) == 2: mask_width = target_shape[0]
        self.mask_width = mask_width
        self.encoder = ConditionalEncoder(condition_dim, target_width, latent_dim, hidden_dim, mask_dim=mask_width)
        self.decoder = ConditionalDecoder(condition_dim, latent_dim, target_width, hidden_dim)
    def forward(self, condition, target, *, noise=None, is_pad=None):
        target_flat = target.reshape(target.shape[0], -1)
        target_mask = None
        if self.target_shape is not None and tuple(target.shape[1:]) != tuple(self.target_shape):
            raise ValueError("target shape does not match model target_shape")
        if is_pad is not None:
            if target.ndim != 3 or is_pad.shape != target.shape[:2]: raise ValueError("is_pad must have shape (B,T)")
            valid = (~is_pad.bool()).unsqueeze(-1).to(target.dtype)
            target_flat = (target * valid).reshape(target.shape[0], -1)
            target_mask = is_pad.to(target.dtype)
        elif self.mask_width:
            target_mask = torch.zeros((target.shape[0], self.mask_width), device=target.device, dtype=target.dtype)
        mu, logvar = self.encoder(condition, target_flat, target_mask)
        z = reparameterize(mu, logvar, noise=noise)
        prediction = self.decoder(condition, z)
        if self.target_shape is not None: prediction = prediction.reshape(target.shape[0], *self.target_shape)
        return prediction, mu, logvar
    def infer(self, condition, *, noise=None):
        shape = (*condition.shape[:-1], self.latent_dim)
        z = torch.randn(shape, device=condition.device, dtype=condition.dtype) if noise is None else noise
        if z.shape != shape: raise ValueError(f"noise must have shape {shape}")
        prediction = self.decoder(condition, z)
        if self.target_shape is not None: prediction = prediction.reshape(condition.shape[0], *self.target_shape)
        return prediction

In [ ]:
cvae = ConditionalVAE(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
train_prediction, mu, logvar = cvae(condition, target, noise=torch.zeros(5, LATENT_DIM))
prior_prediction = cvae.infer(condition, noise=torch.ones(5, LATENT_DIM))
assert train_prediction.shape == prior_prediction.shape == target.shape
assert mu.shape == logvar.shape == (5, LATENT_DIM)
assert not torch.equal(train_prediction, prior_prediction), "Posterior training and prior inference should use their supplied latent values."
print("Training and inference paths have the expected shapes.")

In [ ]:
print("Use: the training path now has the same posterior/prior distinction used by a CVAE policy.")

## Connect the model to a training callback (about 15 min)

Implement a callback with the Unit 3 contract: `loss_fn(model, batch, *, noise=None, time=None) -> (scalar, metrics)`. Use masked action L1 for padded action chunks and Gaussian KL from Unit 3. The supplied batch keys are `condition`, `target`, and, for chunks, `is_pad`.

**Pointers:** Keep `time` for callback compatibility. Pass `noise` to `model` so a check can be repeatable. The returned loss must keep gradients.

In [ ]:
from vla_lab.teaching.training import masked_action_l1, diagonal_gaussian_kl

def cvae_loss_fn(model, batch, *, noise=None, time=None, beta=1.0):
    del time
    prediction, mu, logvar = model(batch["condition"], batch["target"], noise=noise, is_pad=batch.get("is_pad"))
    if prediction.ndim == 3:
        reconstruction = masked_action_l1(prediction, batch["target"], batch["is_pad"])
    else:
        reconstruction = (prediction - batch["target"]).abs().mean()
    kl = diagonal_gaussian_kl(mu, logvar)
    return reconstruction + beta * kl, {"reconstruction": reconstruction, "kl": kl}

In [ ]:
mini_model = ConditionalVAE(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
mini_batch = {"condition": condition, "target": target}
noise = torch.zeros(5, LATENT_DIM)
loss, metrics = cvae_loss_fn(mini_model, mini_batch, noise=noise, beta=0.1)
assert loss.ndim == 0 and torch.isfinite(loss), "Callback must return one finite scalar tensor."
loss.backward()
assert any(parameter.grad is not None for parameter in mini_model.parameters()), "Callback loss must preserve the gradient path."
print("Callback metrics:", {name: float(value.detach()) for name, value in metrics.items()})
# Padded target values cannot change the posterior or objective when random noise is fixed.
chunk_model = ConditionalVAE(CONDITION_DIM, 10, LATENT_DIM, HIDDEN, target_shape=(5, 2))
chunk_mask = torch.tensor([[0,0,0,0,0],[0,0,1,1,1],[0,0,0,1,1],[0,0,0,0,1],[0,1,1,1,1]], dtype=torch.bool)
chunk_data = {"condition": condition, "target": torch.randn(5,5,2), "is_pad": chunk_mask}
chunk_loss, _ = cvae_loss_fn(chunk_model, chunk_data, noise=noise, beta=0.1)
changed = dict(chunk_data, target=chunk_data["target"].clone())
changed["target"][chunk_mask] = 500
changed_loss, _ = cvae_loss_fn(chunk_model, changed, noise=noise, beta=0.1)
torch.testing.assert_close(chunk_loss, changed_loss, msg="Padded action values must not affect CVAE posterior or loss")
print("Chunk mask check passed.")

Use: later units reuse this callback shape for ACT, diffusion, and flow objectives. A caller can pass explicit random values through `noise` and `time` for repeatable checks.

## Experiment — latent size and KL weight (30–45 min)

First fit a small visual example: use a one-dimensional coordinate as the condition and a smooth sinusoidal value with a small ripple as the target. Plot the target curve and predictions from two prior-noise values. Then use the prepared action chunk batch and the supplied callback-compatible `cvae_loss_fn` to compare latent widths 4 and 8 at KL weight 0.1, then compare KL weights 0.1 and 1.0 at latent width 4. This changes one setting at a time.

Keep the train/validation examples, initialization seeds, optimizer, and update count fixed. Record reconstruction error and KL separately. Do not select the setting by training loss alone. Write what changes across prior samples and whether validation supports your prediction.

Set `RUN_ACTION_CHUNK_EXPERIMENT=True` to opt in. The default is `False`. This bounded check uses preloaded local batches and a small network; it does not train ACT or promise a successful policy. `Run All` fits the short CPU toy example above. It does not load local robot data or start a real GPU or robot experiment.

In [ ]:
# Fit a small one-dimensional conditional example and inspect several prior samples.
torch.manual_seed(23)
toy_condition = torch.linspace(-2, 2, 96).unsqueeze(-1)
toy_target = torch.sin(1.5 * toy_condition) + 0.05 * torch.cos(4 * toy_condition)
toy_data = [{"condition": toy_condition[start:start+16], "target": toy_target[start:start+16]} for start in range(0, 96, 16)]
toy_model = ConditionalVAE(1, 1, 2, hidden_dim=32)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=0.01)
from vla_lab.teaching.training import train_loop as reference_train_loop
toy_callback = lambda model, batch, *, noise=None, time=None: cvae_loss_fn(model, batch, noise=noise, time=time, beta=0.05)
toy_history = reference_train_loop(toy_model, toy_data, toy_optimizer, toy_callback, steps=120)
toy_model.eval()
with torch.no_grad():
    prior_zero = toy_model.infer(toy_condition, noise=torch.zeros(96, 2))
    prior_one = toy_model.infer(toy_condition, noise=torch.ones(96, 2))
plt.scatter(toy_condition[:, 0], toy_target[:, 0], s=12, label="target")
plt.plot(toy_condition[:, 0], prior_zero[:, 0], label="prior noise = 0")
plt.plot(toy_condition[:, 0], prior_one[:, 0], label="prior noise = 1")
plt.xlabel("condition")
plt.ylabel("target")
plt.title("Small CVAE example: trained predictions")
plt.legend()
plt.show()
plt.plot([item["loss"] for item in toy_history])
plt.xlabel("Update")
plt.ylabel("Toy CVAE loss")
plt.show()

RUN_ACTION_CHUNK_EXPERIMENT = False
if RUN_ACTION_CHUNK_EXPERIMENT:
    from vla_lab.teaching.live import load_act_batches
    prepared = load_act_batches("bowl_plate", WORKSPACE, batch_size=2, chunk_size=20, train_batches=2, validation_batches=1, seed=31)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def adapt(rows):
        return [{"condition": row["observation.state"].to(device), "target": row["action"].to(device), "is_pad": row["action_is_pad"].to(device)} for row in rows]
    train_rows, validation_rows = adapt(prepared.train), adapt(prepared.validation)
    results = []
    settings = ((4, 0.1), (8, 0.1), (4, 1.0))
    for latent_width, beta in settings:
        row = {"latent_dim": latent_width, "beta": beta, "status": "failed"}
        optimizer = model = None
        try:
            torch.manual_seed(41)
            model = ConditionalVAE(8, 20 * 7, latent_width, hidden_dim=64, target_shape=(20, 7)).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
            torch.manual_seed(1041)
            if device.type == "cuda": torch.cuda.manual_seed_all(1041)
            callback = lambda m, b, *, noise=None, time=None: cvae_loss_fn(m, b, noise=noise, time=time, beta=beta)
            reference_train_loop(model, train_rows, optimizer, callback, steps=20)
            model.eval()
            with torch.no_grad():
                scores = [callback(model, row, noise=torch.zeros(row["condition"].shape[0], latent_width, device=device))[1] for row in validation_rows]
            row["validation_reconstruction"] = sum(item["reconstruction"].item() for item in scores) / len(scores)
            row["validation_kl"] = sum(item["kl"].item() for item in scores) / len(scores)
            row["status"] = "complete"
        except Exception as error:
            row["error"] = f"{type(error).__name__}: {error}"
        results.append(row)
        del optimizer, model
        if device.type == "cuda": torch.cuda.empty_cache()
    print("Prepared split:", prepared.metadata)
    for item in results: print(item)
    print("Compare latent width within beta 0.1 and compare beta within latent width 4. Keep failed rows.")
    print("The evaluation uses zero posterior noise. This short run does not show policy success.")
else:
    print("Skipped action-chunk experiment. Set RUN_ACTION_CHUNK_EXPERIMENT=True to use the bounded local batches.")

## What this model does not include

This small CVAE demonstrates posterior training and prior inference with a conditional MLP. Unit 8 assembles ACT with a Transformer posterior over state and action tokens, camera position information, and action queries. It is a different architecture. Do not treat this MLP as an ACT implementation.